# M4 — Preprocessing, Ekstraksi 100 Fitur Citra & StratifiedGroupKFold Split
**Proyek:** Klasifikasi Penyakit Daun Padi (*Rice Leaf Disease Classification*)

Notebook ini mendokumentasikan tahapan preprocessing citra dan pembagian data bebas *leakage*:
1. Konversi `RGBA` → `RGB` di atas latar putih dan *resize* ke `128×128` piksel.
2. Ekstraksi **100 fitur visual** (32 HSV Hist + 24 RGB Hist + 18 Color Moments + 12 GLCM Texture + 14 Lesion & Sobel Edge).
3. Uji ablasi dampak duplikat MD5 dan *augmentation leakage*.
4. Pembagian data 80:20 menggunakan **StratifiedGroupKFold** berdasarkan `leaf_group` (858 spesimen daun fisik).

In [1]:
import json
import sys
from pathlib import Path
import pandas as pd

sys.path.append(str(Path.cwd().parent))
from src import DATA_PROCESSED_DIR, REPORTS_DIR
from src.preprocessing import get_feature_names

feat_names = get_feature_names()
print("Total fitur yang diekstraksi per citra:", len(feat_names))
print("Contoh 10 fitur pertama:", feat_names[:10])
print("Contoh 10 fitur terakhir:", feat_names[-10:])

with open(REPORTS_DIR / "evaluation_summary.json", "r", encoding="utf-8") as f:
    summary = json.load(f)

print("\n=== Hasil Uji Ablasi Duplikat & Leakage (Random Forest 80:20) ===")
ab = summary["duplicate_ablation"]
print(f"1. Tanpa Hapus Duplikat (5.932 citra, Random Split)     : Acc = {ab['with_duplicates_test_acc']:.4f}, F1 = {ab['with_duplicates_test_f1']:.4f} (LEAKAGE)")
print(f"2. Hapus Duplikat MD5 (4.794 citra, Naive Random Split) : Acc = {ab['naive_dedup_test_acc']:.4f}, F1 = {ab['naive_dedup_test_f1']:.4f} (AUGMENTATION LEAKAGE)")
print(f"3. Hapus Duplikat MD5 + StratifiedGroupKFold Split      : Acc = {ab['group_aware_test_acc']:.4f}, F1 = {ab['group_aware_test_f1']:.4f} (BEBAS LEAKAGE)")

Total fitur yang diekstraksi per citra: 100
Contoh 10 fitur pertama: ['hsv_h_bin_00', 'hsv_h_bin_01', 'hsv_h_bin_02', 'hsv_h_bin_03', 'hsv_h_bin_04', 'hsv_h_bin_05', 'hsv_h_bin_06', 'hsv_h_bin_07', 'hsv_h_bin_08', 'hsv_h_bin_09']
Contoh 10 fitur terakhir: ['ratio_background_white_black', 'exg_excess_green_mean', 'exg_excess_green_std', 'exr_excess_red_mean', 'center_green_ratio', 'center_lesion_ratio', 'sobel_edge_mean', 'sobel_edge_std', 'sobel_edge_p90', 'sobel_high_edge_ratio']

=== Hasil Uji Ablasi Duplikat & Leakage (Random Forest 80:20) ===
1. Tanpa Hapus Duplikat (5.932 citra, Random Split)     : Acc = 1.0000, F1 = 1.0000 (LEAKAGE)
2. Hapus Duplikat MD5 (4.794 citra, Naive Random Split) : Acc = 1.0000, F1 = 1.0000 (AUGMENTATION LEAKAGE)
3. Hapus Duplikat MD5 + StratifiedGroupKFold Split      : Acc = 0.9751, F1 = 0.9733 (BEBAS LEAKAGE)


In [2]:
# Ringkasan hasil pembagian Train vs Test (80:20 StratifiedGroupKFold)
sp = summary["split_info"]
print(f"Train Set : {sp['train_size']} citra ({sp['train_leaf_specimens']} spesimen daun unik)")
print(f"Test Set  : {sp['test_size']} citra ({sp['test_leaf_specimens']} spesimen daun unik)")
print("Distribusi Kelas Train :", sp["train_class_counts"])
print("Distribusi Kelas Test  :", sp["test_class_counts"])

Train Set : 3832 citra (686 spesimen daun unik)
Test Set  : 962 citra (172 spesimen daun unik)
Distribusi Kelas Train : {'Bacterialblight': 1056, 'Tungro': 1048, 'Brownspot': 960, 'Blast': 768}
Distribusi Kelas Test  : {'Bacterialblight': 270, 'Tungro': 260, 'Brownspot': 240, 'Blast': 192}
